In [17]:
import re
from datetime import datetime

with open("./csv/orders.csv", "r", encoding="utf-8") as f:
    text = f.read()

lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
print("HEADER:", lines[0])
print("ROW1  :", lines[1])

HEADER: product_code,order_number,price,number_of_items,order_date
ROW1  : KPT62OVE7BS,913D6853-D43D-7847-31B3-2BBD8B67318E,$449.87,5,03/13/2025


In [18]:
import re

# 1) Order numbers (UUID-Format)
order_numbers = re.findall(r'\b[0-9A-F]{8}(?:-[0-9A-F]{4}){3}-[0-9A-F]{12}\b', text, flags=re.IGNORECASE)

# 2) Product codes (alphanumerisch, Großbuchstaben/Zahlen)
product_codes = re.findall(r'\b[A-Z0-9]{11}\b', text)

# 3) Prices
prices = re.findall(r'\$\d+\.\d{2}', text)
prices_num = [float(p[1:]) for p in prices]

# 4) Order dates (MM/DD/YYYY)
order_dates = re.findall(r'\b\d{2}/\d{2}/\d{4}\b', text)

len(order_numbers), len(product_codes), len(prices), len(order_dates)

(100, 100, 100, 100)

In [19]:
orders_over_500 = []
for ln in lines[1:]:  # ohne header
    m = re.search(r'\$(\d+\.\d{2})', ln)
    if m and float(m.group(1)) > 500:
        orders_over_500.append(ln)

orders_over_500[:10], len(orders_over_500)

(['QNI12WUN3CD,38139EAB-D7B8-7A66-2283-57136C174506,$902.53,3,07/05/2025',
  'XEC54CXI3AG,494166EE-43B1-ED00-576C-A5344A6A2695,$658.46,7,01/30/2025',
  'IMF54TDN1JR,6EB66922-DC64-1865-BA11-8391D91481D5,$655.72,8,04/04/2025',
  'CTS23VKM0HQ,75E0D2B7-CCE8-20A7-68BB-2EAB73599AC9,$826.02,2,08/12/2025',
  'QET08DJU7PM,69899781-1889-9979-4785-1CF8DC64F606,$508.56,8,04/08/2025',
  'COO14PMT4VM,F459BA97-4EA3-4724-34A9-267E57CBF533,$746.35,9,03/28/2025',
  'NHU70QWB4NF,80B3E838-9416-082B-3594-BB6B86DAE874,$854.44,8,08/29/2025',
  'KXW13GBV6YU,94894E91-1364-4897-53DB-4C35F8DE4B11,$594.33,7,05/15/2025',
  'QBT75SEU9EV,E604E618-FA56-83E8-3A11-881220EA5656,$636.91,2,06/01/2025',
  'EYQ86OKC0US,5BDA4B85-64FB-4F99-71BC-CB101A5C51F7,$850.72,3,02/28/2025'],
 55)

In [20]:
def mmddyyyy_to_ddmmyyyy(match):
    mm, dd, yyyy = match.group(1), match.group(2), match.group(3)
    return f"{dd}/{mm}/{yyyy}"

text_ddmmyyyy = re.sub(r'\b(\d{2})/(\d{2})/(\d{4})\b', mmddyyyy_to_ddmmyyyy, text)

print("Before:", lines[1])
print("After :", text_ddmmyyyy.splitlines()[1])

Before: KPT62OVE7BS,913D6853-D43D-7847-31B3-2BBD8B67318E,$449.87,5,03/13/2025
After : KPT62OVE7BS,913D6853-D43D-7847-31B3-2BBD8B67318E,$449.87,5,13/03/2025


In [21]:
# Extract number_of_items from each line (5-column CSV)
items_per_order = []
for ln in lines[1:]:
    m = re.search(r'^\s*[^,]+,[^,]+,\$\d+\.\d{2},(\d+),\d{2}/\d{2}/\d{4}\s*$', ln)
    if m:
        items_per_order.append((int(m.group(1)), ln))

max_items = max(n for n, _ in items_per_order)
max_item_orders = [ln for n, ln in items_per_order if n == max_items]

max_items, max_item_orders[:10], len(max_item_orders)

(10,
 ['POO81PLQ3MA,CADB9B36-D5B0-7745-9ADA-8BAC1C0E5B1D,$411.14,10,06/12/2025',
  'BRF08UXC4EZ,9979E439-F489-7872-572C-99BFE129B231,$595.82,10,01/10/2025',
  'SRD14VWO7RT,937B2801-CD82-1735-68A1-661AABE812A9,$85.36,10,07/29/2025',
  'CYW04QOY9QH,1C6AEF1D-E0D7-4489-9672-E65B5B8A4392,$341.99,10,04/14/2025',
  'RBI93QCQ5EX,6FF65E38-5FC0-7436-E0D2-AEB082ACA3FA,$553.15,10,07/05/2025',
  'KCP17OKT6WR,A49449D1-29DE-D7C5-42B1-D56381F8158B,$510.46,10,06/08/2025'],
 6)

In [22]:
priced_lines = []
for ln in lines[1:]:
    m = re.search(r'\$(\d+\.\d{2})', ln)
    if m:
        priced_lines.append((float(m.group(1)), ln))

min_price = min(p for p, _ in priced_lines)
cheapest_orders = [ln for p, ln in priced_lines if p == min_price]

min_price, cheapest_orders[:10], len(cheapest_orders)

(11.59,
 ['FDV51RUV6XV,C8EE58CD-E0ED-73CF-852C-CEC6A91C345C,$11.59,8,08/27/2025'],
 1)